[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/MachineLearningKurs/blob/main/set14-neuronale-netze-nn2/uebungen/01_uebung_kleines_cnn.ipynb)

**In Colab:** Zuerst die folgende Setup-Zelle ausführen. [Anleitung zum Start](https://colab.research.google.com/github/Y-Robin/MachineLearningKurs/blob/main/set1-python-grundlagen/00_start_mit_google_colab.ipynb).


In [ ]:
# Nur in Google Colab: Repository, Bibliotheken und Arbeitsordner einrichten.
import importlib.util

if (importlib.util.find_spec("google") is not None
        and importlib.util.find_spec("google.colab") is not None):
    from pathlib import Path
    import runpy
    import subprocess

    repo = Path("/content/MachineLearningKurs")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "main",
                               "https://github.com/Y-Robin/MachineLearningKurs.git", str(repo)])
    runpy.run_path(str(repo / "colab_setup.py"))["setup"](
        'set14-neuronale-netze-nn2/uebungen/01_uebung_kleines_cnn.ipynb', cpu=True)


# Miniübung: Das CNN lesen

**Ca. 5–10 Minuten.** Hier geht es um Formen und Bausteine, nicht um ein weiteres Training.
Die Bilder sind Beispieldaten zum Anschauen, keine neue Evaluation.

In [1]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
torch.set_num_threads(2)
# Ohne .cuda() oder .to("cuda") bleiben Modelle und Tensoren auf der CPU.
print("Gerät:", torch.tensor(0).device)

Gerät: cpu


In [2]:
daten = load_digits()
bilder = torch.tensor(daten.images / 16.0, dtype=torch.float32)
bilder = bilder.unsqueeze(1)  # (1797, 8, 8) -> (1797, 1, 8, 8)
labels = torch.tensor(daten.target, dtype=torch.long)
print("Bilder:", bilder.shape)
print("Erstes Label:", labels[0].item())

Bilder: torch.Size([1797, 1, 8, 8])
Erstes Label: 0


## 1. Formen vorhersagen

Das Netz unten verarbeitet zwei Bilder. Notiere **vor dem Ausführen** die
Form nach jeder Schicht. Warum braucht `Linear` genau 64 Eingaben?

In [3]:
modell = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, padding=1),  # 1 x 8 x 8 -> 4 x 8 x 8
    nn.ReLU(),
    nn.MaxPool2d(2),                          # 4 x 8 x 8 -> 4 x 4 x 4
    nn.Flatten(),                            # 4 x 4 x 4 -> 64
    nn.Linear(64, 10),                        # 64 -> 10 Ziffernscores
)
x = bilder[:2]
for schicht in modell:
    x = schicht(x)
    print(type(schicht).__name__, tuple(x.shape))

Conv2d (2, 4, 8, 8)
ReLU (2, 4, 8, 8)
MaxPool2d (2, 4, 4, 4)
Flatten (2, 64)
Linear (2, 10)


## 2. Pooling von Hand

Berechne das Ergebnis für jeden 2×2-Block, bevor du die Zelle ausführst.
Welche Information geht verloren?

In [4]:
beispiel = torch.tensor([[[[1., 2., 0., 4.],
                          [3., 0., 1., 2.],
                          [0., 1., 5., 2.],
                          [2., 4., 1., 0.]]]])
print(nn.MaxPool2d(2)(beispiel))

tensor([[[[3., 4.],
          [4., 5.]]]])


## 3. Zwei statt vier Filter

Kopiere die Netzdefinition und ändere `Conv2d` auf zwei Filter.
Passe auch `Linear` an. Prüfe an zwei Bildern, dass weiter zehn Scores entstehen.

In [5]:
# Hier dein Netz eintragen.

## 4. Training erklären

Welche Zeile aus Notebook 01 berechnet Gradienten, welche verändert Gewichte?